In [13]:
import io
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import torch


In [28]:
# ============================================================================
# CONFIGURATION - ADJUST THESE FOR YOUR NEEDS
# ============================================================================

USE_SAMPLE_SIZE = 5000   # REDUCED! 5k samples only
USE_FAST_MODEL = True    # True = DistilBERT (fast), False = BERT (slow)
NUM_EPOCHS = 1           # Just 1 epoch
MAX_STEPS = 200          # REDUCED! Only 200 steps (~10-15 min)
BATCH_SIZE = 32          # INCREASED! Larger batches = fewer steps


In [29]:
# ============================================================================
# 1. LOAD DATASET
# ============================================================================

# Replace this with your actual CSV loading:
# df = pd.read_csv('your_file.csv')

# For demo purposes (replace with your data):
csv_text = """ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance
1,Reconstructing Subject-Specific Effect Maps," Predictive models allow subject-specific inference when analyzing disease related alterations in neuroimaging data. Given......",1,0,0,0,0,0"""

# df = pd.read_csv(io.StringIO(csv_text))
df = pd.read_csv("train.csv")

print(f"Original dataset size: {len(df)} samples")

# ============================================================================
# OPTIMIZATION 1: Sample the dataset for faster training
# ============================================================================

if USE_SAMPLE_SIZE and USE_SAMPLE_SIZE < len(df):
    df = df.sample(n=USE_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Using sample of {USE_SAMPLE_SIZE} samples for faster training")
else:
    print(f"Using full dataset: {len(df)} samples")


Original dataset size: 20972 samples
Using sample of 5000 samples for faster training


In [30]:
# ============================================================================
# 2. PREPARE DATA
# ============================================================================

label_columns = [
    'Computer Science', 'Physics', 'Mathematics', 
    'Statistics', 'Quantitative Biology', 'Quantitative Finance'
]

texts = df['ABSTRACT'].tolist()
labels = df[label_columns].values.astype(np.float32)

print(f"\nLabel distribution:\n{df[label_columns].sum()}")



Label distribution:
Computer Science        2017
Physics                 1453
Mathematics             1361
Statistics              1254
Quantitative Biology     157
Quantitative Finance      55
dtype: int64


In [32]:
# ============================================================================
# 4. TOKENIZATION
# ============================================================================

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        max_length=128,
        padding='max_length',  # Pad to max_length for faster training
        truncation=True
    )

train_dataset = Dataset.from_dict({
    'text': train_texts,
    'labels': train_labels.tolist()
})

eval_dataset = Dataset.from_dict({
    'text': eval_texts,
    'labels': eval_labels.tolist()
})

print("\nTokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])



Tokenizing datasets...


Map: 100%|██████████| 1000/1000 [00:00<00:00, 10869.62 examples/s]


In [ ]:
# ============================================================================
# 5. MODEL CONFIGURATION
# ============================================================================

# 5. MODEL CONFIGURATION
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
# ============================================================================
# 6. COMPUTE METRICS
# ============================================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.sigmoid(torch.tensor(logits)).numpy()
    binary_predictions = (predictions > 0.5).astype(int)
    
    macro_f1 = f1_score(labels, binary_predictions, average='macro', zero_division=0)
    
    try:
        macro_roc_auc = roc_auc_score(labels, predictions, average='macro')
    except ValueError:
        macro_roc_auc = 0.0
    
    return {
        'macro_f1': macro_f1,
        'macro_roc_auc': macro_roc_auc
    }

# ============================================================================
# OPTIMIZATION 3: Efficient training arguments
# ============================================================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=3e-5,  # CHANGED: Slightly higher
    weight_decay=0.01,
    warmup_steps=20,  # CHANGED: REDUCED from 100 to 20
    eval_strategy="steps",
    eval_steps=100,  # CHANGED: REDUCED from 500 to 100
    save_strategy="steps",
    save_steps=100,  # CHANGED: REDUCED from 500 to 100
    save_total_limit=1,  # CHANGED: Keep only 1 checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir='./logs',
    logging_steps=20,  # CHANGED: REDUCED from 100 to 20
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=42,
    report_to="none"
)

print("\n" + "="*70)
print("TRAINING CONFIGURATION")
print("="*70)
print(f"Model: {model_name}")
print(f"Training samples: {len(train_texts)}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Max steps: {MAX_STEPS if MAX_STEPS > 0 else 'No limit'}")
print(f"Batch size: {BATCH_SIZE}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Estimated training time: ", end="")

# Rough time estimation
steps_per_epoch = len(train_texts) // BATCH_SIZE
total_steps = min(steps_per_epoch * NUM_EPOCHS, MAX_STEPS) if MAX_STEPS > 0 else steps_per_epoch * NUM_EPOCHS
seconds_per_step = 0.5 if torch.cuda.is_available() else 2.0  # GPU vs CPU
estimated_minutes = (total_steps * seconds_per_step) / 60
print(f"~{estimated_minutes:.1f} minutes")
print("="*70)



TRAINING CONFIGURATION
Model: distilbert-base-uncased
Training samples: 4000
Epochs: 1
Max steps: 200
Batch size: 32
GPU available: False
Estimated training time: ~4.2 minutes


In [36]:
# ============================================================================
# 7. TRAIN
# ============================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

print("\nStarting training...\n")
trainer.train()


C:\Users\pc\AppData\Local\Temp\ipykernel_25416\87638827.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting training...



c:\Users\pc\Secondary-Documents\Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Macro F1,Macro Roc Auc
100,0.282500,0.271686,0.466730,0.822027
200,0.225900,0.238172,0.524734,0.927300


c:\Users\pc\Secondary-Documents\Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=200, training_loss=0.3218455195426941, metrics={'train_runtime': 836.1764, 'train_samples_per_second': 7.654, 'train_steps_per_second': 0.239, 'total_flos': 211962957004800.0, 'train_loss': 0.3218455195426941, 'epoch': 1.6})

In [37]:
# ============================================================================
# 8. EVALUATE
# ============================================================================

print("\n" + "="*70)
print("FINAL EVALUATION RESULTS")
print("="*70)

eval_results = trainer.evaluate()

print(f"\n{'Metric':<30} {'Value':<10}")
print("-" * 45)
print(f"{'Macro-Averaged F1-Score':<30} {eval_results['eval_macro_f1']:.4f}")
print(f"{'Macro-Averaged ROC-AUC':<30} {eval_results['eval_macro_roc_auc']:.4f}")
print(f"{'Evaluation Loss':<30} {eval_results['eval_loss']:.4f}")
print("-" * 45)



FINAL EVALUATION RESULTS


c:\Users\pc\Secondary-Documents\Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Metric                         Value     
---------------------------------------------
Macro-Averaged F1-Score        0.5247
Macro-Averaged ROC-AUC         0.9273
Evaluation Loss                0.2382
---------------------------------------------


In [38]:
# ============================================================================
# 9. SAMPLE PREDICTIONS
# ============================================================================

print("\n" + "="*70)
print("SAMPLE PREDICTIONS")
print("="*70)

predictions = trainer.predict(eval_dataset)
pred_probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
pred_labels = (pred_probs > 0.5).astype(int)

for idx in range(min(3, len(eval_texts))):
    print(f"\nSample {idx + 1}:")
    print(f"Text: {eval_texts[idx][:100]}...")
    print(f"True:      {eval_labels[idx].astype(int)}")
    print(f"Predicted: {pred_labels[idx]}")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\nModel saved to: ./results")
print(f"Checkpoints saved every 500 steps to: ./results/checkpoint-*")
print(f"\nTo resume from checkpoint if interrupted:")
print(f"trainer.train(resume_from_checkpoint='./results/checkpoint-XXXX')")


SAMPLE PREDICTIONS


c:\Users\pc\Secondary-Documents\Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Sample 1:
Text:   In this paper, we consider the problem of attack-resilient state estimation,
that is to reliably e...
True:      [1 0 1 0 0 0]
Predicted: [1 0 0 0 0 0]

Sample 2:
Text:   In this paper, we consider several compression techniques for the language
modeling problem based ...
True:      [1 0 0 1 0 0]
Predicted: [1 0 0 0 0 0]

Sample 3:
Text:   Obtaining accurate estimates of satellite drag coefficients in low Earth
orbit is a crucial compon...
True:      [0 0 0 1 0 0]
Predicted: [0 0 0 0 0 0]

TRAINING COMPLETE!

Model saved to: ./results
Checkpoints saved every 500 steps to: ./results/checkpoint-*

To resume from checkpoint if interrupted:
trainer.train(resume_from_checkpoint='./results/checkpoint-XXXX')
